# Multiclass Classic ML Inference

This notebook runs multiclass inference for one `.data` file using artifacts from `multilabel_ml_train.ipynb`:
- saved model (`.cbm` or `.joblib`)
- saved preprocessing artifacts (`preprocessing.joblib`)
- metadata (`metadata.json`) with class mapping 0..3 -> label_00..label_03

In [ ]:
from pathlib import Path
import sys
import json

import joblib
import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'tools').exists() and (project_root.parent / 'tools').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.csi_parser import Parser
from tools.filters import median_filter

In [ ]:
def skewness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 3))


def kurtosis_excess(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 4) - 3.0)


def spectral_features(signal: np.ndarray) -> tuple[float, float, float, float, float]:
    x = np.asarray(signal, dtype=np.float64)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    spectrum = np.fft.rfft(x)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    power_sum = power.sum()
    if power_sum <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    centroid = float((freqs * power).sum() / power_sum)
    spread = float(np.sqrt(((freqs - centroid) ** 2 * power).sum() / power_sum))

    p_norm = power / power_sum
    eps = 1e-12
    entropy = float(-(p_norm * np.log2(p_norm + eps)).sum())

    dominant_idx = int(np.argmax(power))
    dominant_freq = float(freqs[dominant_idx])
    rolloff_threshold = 0.85 * power_sum
    rolloff = float(freqs[np.searchsorted(np.cumsum(power), rolloff_threshold)])

    return centroid, spread, entropy, dominant_freq, rolloff


def build_unit_signal(df: pd.DataFrame, window: int) -> np.ndarray:
    amp_matrix = np.stack(df['amplitude'].to_numpy(), axis=0).astype(np.float64)
    raw_time_signal = amp_matrix.mean(axis=1)
    return median_filter(raw_time_signal, window)


def build_feature_row(df: pd.DataFrame, signal: np.ndarray, global_min: float, global_max: float) -> dict[str, float]:
    den = global_max - global_min
    if den <= 0:
        raise ValueError('Invalid scaling parameters: global_max must be > global_min')

    scaled_signal = (signal - global_min) / den
    scaled_signal = np.clip(scaled_signal, 0.0, 1.0)

    mean_val = float(np.mean(scaled_signal))
    std_val = float(np.std(scaled_signal))
    median_val = float(np.median(scaled_signal))
    min_val = float(np.min(scaled_signal))
    max_val = float(np.max(scaled_signal))
    q25 = float(np.percentile(scaled_signal, 25))
    q75 = float(np.percentile(scaled_signal, 75))
    iqr = q75 - q25
    range_val = max_val - min_val
    rms = float(np.sqrt(np.mean(scaled_signal ** 2)))
    energy = float(np.sum(scaled_signal ** 2))
    zcr = float(np.mean(np.abs(np.diff(np.signbit(scaled_signal - mean_val)).astype(np.int8))))

    sk = skewness(scaled_signal)
    kt = kurtosis_excess(scaled_signal)
    sc, ss, se, dom_freq, rolloff = spectral_features(scaled_signal)

    rssi_dbm = float(df['rssi_dbm'].median())

    return {
        'mean': mean_val,
        'std': std_val,
        'median': median_val,
        'skew': sk,
        'kurtosis': kt,
        'spectral_centroid': sc,
        'spectral_spread': ss,
        'spectral_entropy': se,
        'dominant_freq': dom_freq,
        'spectral_rolloff_85': rolloff,
        'min': min_val,
        'max': max_val,
        'q25': q25,
        'q75': q75,
        'iqr': iqr,
        'range': range_val,
        'rms': rms,
        'energy': energy,
        'zcr': zcr,
        'rssi_dbm': rssi_dbm,
    }


def load_model(model_path: Path, model_type: str | None):
    resolved_type = model_type
    if resolved_type is None:
        resolved_type = 'catboost' if model_path.suffix.lower() == '.cbm' else 'sklearn'

    if resolved_type == 'catboost':
        from catboost import CatBoostClassifier

        model = CatBoostClassifier()
        model.load_model(str(model_path))
        return model, resolved_type

    if resolved_type == 'sklearn':
        model = joblib.load(model_path)
        return model, resolved_type

    raise ValueError(f'Unsupported model type: {resolved_type}')

In [ ]:
artifacts_dir = project_root / 'artifacts' / 'multilabel_ml'
metadata_path = artifacts_dir / 'metadata.json'

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
model_path = Path(metadata['model_path'])
preprocessing_path = Path(metadata['preprocessing_path'])

# Example file for inference
data_file = project_root / 'wifi_data_set_fixed' / 'id_person_03' / 'label_02' / 'test_18' / 'test18__dev1_64_E8_33_57_AA_F4.data'

print(f'metadata: {metadata_path}')
print(f'model: {model_path}')
print(f'preprocessing: {preprocessing_path}')
print(f'data_file: {data_file}')

In [ ]:
preproc_bundle = joblib.load(preprocessing_path)

median_width = int(preproc_bundle['median_width'])
global_min = float(preproc_bundle['global_min'])
global_max = float(preproc_bundle['global_max'])
feature_cols = list(preproc_bundle['feature_cols'])
feature_min = preproc_bundle.get('feature_min')
feature_max = preproc_bundle.get('feature_max')
scaler_pca = preproc_bundle['scaler_pca']
pca = preproc_bundle['pca']
k_95 = int(preproc_bundle['k_95'])
model_type = preproc_bundle.get('model_type')

class_mapping = metadata.get('class_mapping', {})

model, resolved_model_type = load_model(model_path, model_type)

df = Parser(data_file).parse()
signal = build_unit_signal(df, median_width)
feature_row = build_feature_row(df, signal, global_min, global_max)

feature_df = pd.DataFrame([feature_row])
X = feature_df[feature_cols].to_numpy(dtype=np.float64)

if feature_min is not None and feature_max is not None:
    fmin = np.asarray(feature_min, dtype=np.float64)
    fmax = np.asarray(feature_max, dtype=np.float64)
    if fmin.shape == X.shape[1:] and fmax.shape == X.shape[1:]:
        X = np.clip(X, fmin, fmax)

X_std = scaler_pca.transform(X)
X_pca_95 = pca.transform(X_std)[:, :k_95]

pred = int(model.predict(X_pca_95)[0])

if hasattr(model, 'predict_proba'):
    proba = np.asarray(model.predict_proba(X_pca_95)[0], dtype=np.float64)
    if hasattr(model, 'classes_'):
        proba_classes = [int(c) for c in model.classes_]
    else:
        proba_classes = list(range(len(proba)))
    proba_by_class = {str(c): float(p) for c, p in zip(proba_classes, proba)}
else:
    proba_by_class = {}

predicted_label_name = class_mapping.get(str(pred), f'class_{pred}')

result = {
    'data_file': str(data_file),
    'predicted_class': pred,
    'predicted_label_name': predicted_label_name,
    'probabilities_by_class': proba_by_class,
    'model_type': resolved_model_type,
    'k_95': k_95,
    'n_packets': int(len(df)),
}

print(json.dumps(result, indent=2, ensure_ascii=False))
result

## Notes

- This notebook uses the same preprocessing as training (median filter, train-derived scaling bounds, clipping, StandardScaler, PCA).
- Classes are interpreted using `artifacts/multilabel_ml/metadata.json`.